In [ ]:
import pandas as pd
from pathlib import Path

path = Path.cwd() / "task_2_data_ex.xlsx"

data = pd.read_excel(path)
display(data.head())



## ANALYSIS
(For my own tests)

In [ ]:
produced_materials = data.groupby("produced_material")
display(produced_materials.describe())

In [ ]:
produced_materials = data.groupby("produced_material_production_type")
display(produced_materials.describe())

In [ ]:
produced_materials = data.groupby("produced_material_release_type")
display(produced_materials.describe())

In [ ]:
# produced_materials = data.groupby(["component_material_production_type", "component_material_release_type"])
# display(produced_materials.describe())
produced_materials = data.groupby(["component_material_release_type"])
display(produced_materials.describe())



In [ ]:
y_fin = data.loc[data["produced_material"] == 10000]
display(y_fin)
# y_comp1 = data.loc[data["produced_material"] == 50000]
# display(y_comp1)
# y_comp2 = data.loc[(data["produced_material"] == 80070) | (data["produced_material"] == 90001)]
# display(y_comp2)


## ACTUAL SOLUTION

The task is to gather all unique materials present in the dataset and then show their production hierarchy,
by finding their final material (product) and their component.
Final material id's exist only in **produced_material** column, while base components exist only in **component_material** column.


In [ ]:
def get_unique_materials(data: pd.DataFrame) -> set:
    """
    Returns all unique material ID's present in the dataset.
    """
    return set(data["produced_material"]).union(data["component_material"])

#print(get_unique_materials(data))

def get_material_data(data: pd.DataFrame, material_id: int):
    """ 
    Returns the required data for 
    """
    material_data = data.loc[data["produced_material"] == material_id]
    columns = ["year","month","produced_material", "produced_material_production_type", "produced_material_release_type", "produced_material_quantity"]
    return material_data[columns]

def get_materials_final(data: pd.DataFrame, material_id: int, previous_mats: list[int] | None = None) -> pd.DataFrame:
    """
    Returns a tuple of data of the final component of a material with given ID and additionally its production path 
    """
    if previous_mats is None:
        previous_mats = []
    if material_id in previous_mats:
        raise Exception(f"Data error - material {material_id} has a loop in its production line")
    previous_mats.append(material_id)

    component_data = data.loc[data["component_material"] == material_id]
    columns = ["year","month","produced_material", "produced_material_production_type", "produced_material_release_type", "produced_material_quantity"]
    # if final material:
    if component_data.empty:
        produced_data = data.loc[(data["produced_material"] == material_id) & (data["produced_material_release_type"] == "FIN")]
        
        return produced_data[columns], previous_mats
    else:
        produced_id = component_data["produced_material"].unique()
        #print(produced_id)
        return get_materials_final(data, int(produced_id[0]), previous_mats)
        

    


def get_component_data(data: pd.DataFrame, material_id: int) -> pd.DataFrame:
    """
    Returns the data of the material's component. Returns empty DataFrame for base materials, since they do not have components.

    """
    component_data = data.loc[data["produced_material"] == material_id]
    columns = ["year", "month", "component_material", "component_material_production_type", "component_material_release_type", "component_material_quantity"]
    
    
    return component_data[columns]


def merge_and_sum_yearly(data: pd.DataFrame, material_id: int) -> pd.DataFrame:
    """
    Merges a material's production data with its component data and
    sums the quantities per year.
    """
    material_data = get_material_data(data, material_id).rename(columns={
        "produced_material":                 "Prod_Material_id",
        "produced_material_release_type":    "Prod_material_Release_type",
        "produced_material_production_type": "Prod_material_production_type",
        "produced_material_quantity":        "Prod_material_production_quantity",
    })
    component_data = get_component_data(data, material_id)

    merged = material_data.merge(component_data, on=["year", "month"], how="inner")

    group_cols = [
        "year",
        "Prod_Material_id", "Prod_material_production_type", "Prod_material_Release_type",
        "component_material", "component_material_production_type", "component_material_release_type",
    ]
    yearly = merged.groupby(group_cols, as_index=False).agg(
        Prod_material_production_quantity=("Prod_material_production_quantity", "sum"),
        component_material_quantity=("component_material_quantity", "sum"),
    )
    return yearly
    




In [ ]:
def build_material_summary(data: pd.DataFrame, material_id: int) -> pd.DataFrame:
    """
    Yearly production summary for a material, per plant:
    its final (FIN) product, the material itself, and each component
    it is produced from. Quantities are summed over each year.
    """
    # rows where this material is the produced item -> carries Prod + Component levels
    prod_rows = data.loc[data["produced_material"] == material_id].copy()

    # climb the production line to the final (FIN) material
    _, path = get_materials_final(data, material_id)
    fin_id = path[-1]

    # rows for the final product -> Fin level (renamed up front to avoid a clash on merge)
    fin_cols = ["year", "month", "plant_id",
                "produced_material", "produced_material_production_type",
                "produced_material_release_type", "produced_material_quantity"]
    fin_rows = (data.loc[data["produced_material"] == fin_id, fin_cols]
                    .drop_duplicates()
                    .rename(columns={
                        "produced_material":                 "fin_material_id",
                        "produced_material_release_type":    "fin_material_release_type",
                        "produced_material_production_type": "fin_material_production_type",
                        "produced_material_quantity":        "fin_production_quantity",
                    }))

    merged = prod_rows.merge(fin_rows, on=["year", "month", "plant_id"], how="left").rename(columns={
        "plant_id":                          "plant",
        "year":                              "year",
        "produced_material":                 "prod_material_id",
        "produced_material_release_type":    "prod_material_release_type",
        "produced_material_production_type": "prod_material_production_type",
        "produced_material_quantity":        "prod_material_production_quantity",
        "component_material":                "component_id",
        "component_material_release_type":   "component_material_release_type",
        "component_material_production_type":"component_material_production_type",
        "component_material_quantity":       "component_consumption_quantity",
    })

    group_cols = [
        "plant", "year",
        "fin_material_id", "fin_material_release_type", "fin_material_production_type",
        "prod_material_id", "prod_material_release_type", "prod_material_production_type",
        "component_id", "component_material_release_type", "component_material_production_type",
    ]
    # dropna=False: base components have NaN production_type; without this those
    # component rows would be silently dropped from the summary.
    summary = merged.groupby(group_cols, as_index=False, dropna=False).agg(
        fin_production_quantity=("fin_production_quantity", "sum"),
        prod_material_production_quantity=("prod_material_production_quantity", "sum"),
        component_consumption_quantity=("component_consumption_quantity", "sum"),
    )

    column_order = [
        "plant",
        "fin_material_id", "fin_material_release_type", "fin_material_production_type", "fin_production_quantity",
        "prod_material_id", "prod_material_release_type", "prod_material_production_type", "prod_material_production_quantity",
        "component_id", "component_material_release_type", "component_material_production_type", "component_consumption_quantity",
        "year",
    ]
    return summary[column_order]

In [ ]:
# material_ID = 50000
# x = get_material_data(data, material_ID)
# print(f"Production data of material {material_ID}")
# display(x)

# material_ID = 50000
# x, path = get_materials_final(data, material_ID)
# print(f"Final material data of {material_ID} with production path: {path}")
# display(x)    

# material_ID = 50000
# print(f"Component data of material {material_ID}")
# display(get_component_data(data, material_ID))

# material_ID = 50000
# print(f"Full table for material {material_ID}:")
# display(build_material_summary(data, material_ID))

In [204]:
def get_all_material_summary(data: pd.DataFrame) -> pd.DataFrame:
    produced_materials = data["produced_material"].unique()
    summaries = [build_material_summary(data, int(mat)) for mat in produced_materials]
    return pd.concat(summaries, ignore_index=True)

display(get_all_material_summary(data))

,plant,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity,prod_material_id,prod_material_release_type,prod_material_production_type,prod_material_production_quantity,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity,year
0,RLT_10,10000,FIN,8002,11708.0,10000,FIN,8002,11708.0,50000,PROD,8002.0,11708.0,2024
1,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,80070,PROD,8007.0,11303.0,2024
2,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90000,ADD,NaN,598.0,2024
3,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90001,ADD,NaN,242.0,2024
4,RLT_10,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,80010,PROD,8001.0,41769.0,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,RLT_14,10009,FIN,8002,12023.0,80079,PROD,8007,10751.0,90048,ADD,NaN,122.0,2024
106,RLT_14,10009,FIN,8002,12023.0,80019,PROD,8001,21300.0,80009,PROD,8000.0,24730.0,2024
107,RLT_14,10009,FIN,8002,12023.0,80019,PROD,8001,21300.0,90049,ADD,NaN,1214.0,2024
108,RLT_14,10009,FIN,8002,12023.0,80009,PROD,8000,24238.0,70009,RM,NaN,30624.0,2024
